# Lab | Model Deployment

This notebook walks through the complete ML deployment workflow: train a classifier, serialize it, test a REST API, and reflect on production concerns.

## Imports

In [21]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

---
## Task 1: Train & Serialize

We load the Iris dataset (150 samples, 4 features, 3 species) and train a `RandomForestClassifier`. The model is then serialized with `joblib` for later use in the API.

In [22]:
# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

print(f"Dataset shape: {X.shape}")
print(f"Classes: {iris.target_names}")
print(f"Features: {iris.feature_names}")

Dataset shape: (150, 4)
Classes: ['setosa' 'versicolor' 'virginica']
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [23]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 120, Test size: 30


In [24]:
# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Test Accuracy: 1.0000

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



The model achieves **100% accuracy** on the 30-sample test set. Iris is a well-separated dataset, so this is expected for a Random Forest with 100 trees.

In [25]:
# Serialize model and target names
joblib.dump(model, "model.joblib")
joblib.dump(list(iris.target_names), "target_names.joblib")
print("Saved model.joblib and target_names.joblib")

Saved model.joblib and target_names.joblib


In [26]:
# Round-trip verification
model_loaded = joblib.load("model.joblib")
y_pred_loaded = model_loaded.predict(X_test)

identical = np.array_equal(y_pred, y_pred_loaded)
print(f"Predictions identical after reload: {identical}")
assert identical, "Round-trip check FAILED!"

Predictions identical after reload: True


The serialized model produces bit-for-bit identical predictions — the joblib round-trip is verified.

---
## Task 3: Test the API

> **Before running this section**, open a terminal and start the Flask server:  
> ```bash
> python app.py
> ```

We test the four endpoint scenarios: health check, valid single prediction, error handling, and batch prediction.

### 3.1 Health Check

In [27]:
BASE_URL = "http://localhost:5000"

resp = requests.get(f"{BASE_URL}/health")
print(f"Status code: {resp.status_code}")
print(f"Response:    {resp.json()}")
assert resp.status_code == 200
assert resp.json()["status"] == "healthy"
print("✓ Health check passed")

Status code: 200
Response:    {'status': 'healthy'}
✓ Health check passed


The `/health` endpoint returns `200 {"status": "healthy"}` — a standard liveness signal used by load balancers and container orchestrators.

### 3.2 Single Prediction

In [28]:
sample = {"features": [5.1, 3.5, 1.4, 0.2]}   # classic setosa sample

resp = requests.post(f"{BASE_URL}/predict", json=sample)
print(f"Status code:     {resp.status_code}")
result = resp.json()
print(f"Predicted class: {result['predicted_class']}")
print(f"Probabilities:   {result['probabilities']}")
assert resp.status_code == 200
print("✓ Single prediction passed")

Status code:     200
Predicted class: setosa
Probabilities:   {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}
✓ Single prediction passed


The API correctly identifies the sample as *setosa* with high confidence. The JSON response includes both the class label and per-class probabilities, which is useful for downstream decision-making.

### 3.3 Error Handling

In [29]:
# Error 1: Missing 'features' key
resp = requests.post(f"{BASE_URL}/predict", json={"data": [1, 2, 3, 4]})
print(f"[Missing key]  status={resp.status_code}  msg={resp.json()}")
assert resp.status_code == 400

# Error 2: Wrong number of features (3 instead of 4)
resp = requests.post(f"{BASE_URL}/predict", json={"features": [5.1, 3.5, 1.4]})
print(f"[Wrong count]  status={resp.status_code}  msg={resp.json()}")
assert resp.status_code == 400

# Error 3: Non-numeric values
resp = requests.post(f"{BASE_URL}/predict", json={"features": ["a", "b", "c", "d"]})
print(f"[Non-numeric]  status={resp.status_code}  msg={resp.json()}")
assert resp.status_code == 400

print("✓ All 3 error cases correctly returned 400")

[Missing key]  status=400  msg={'error': "Missing 'features' key in request body."}
[Wrong count]  status=400  msg={'error': "Field 'features' must be a list of exactly 4 numeric values."}
[Non-numeric]  status=400  msg={'error': "All features must be numeric. Got: 'a'"}
✓ All 3 error cases correctly returned 400


All three malformed requests return `400 Bad Request` with a descriptive error message:
1. **Missing key** — the `features` field is absent.
2. **Wrong count** — only 3 values provided instead of 4.
3. **Non-numeric** — string values cannot be passed to the model.

Returning explicit error messages (rather than a generic 500) is critical for API usability.

### 3.4 Batch Prediction

In [30]:
# Pick 5 test samples and get local ground truth
batch_X = X_test[:5].tolist()
local_preds = model.predict(np.array(batch_X))
target_names = joblib.load("target_names.joblib")
local_labels = [target_names[p] for p in local_preds]

# Send to API
resp = requests.post(f"{BASE_URL}/predict_batch", json={"samples": batch_X})
assert resp.status_code == 200
api_labels = [r["predicted_class"] for r in resp.json()["predictions"]]

print(f"Local predictions: {local_labels}")
print(f"API   predictions: {api_labels}")
assert local_labels == api_labels
print("✓ Batch predictions match local model")

Local predictions: [np.str_('versicolor'), np.str_('setosa'), np.str_('virginica'), np.str_('versicolor'), np.str_('versicolor')]
API   predictions: ['versicolor', 'setosa', 'virginica', 'versicolor', 'versicolor']
✓ Batch predictions match local model


The batch endpoint returns the same predictions the local model does — confirming the serialized model loaded in the API is equivalent.

---
## Task 4: Reflection

### What would need to change for a production deployment?

The development Flask server (`app.run(debug=True)`) is single-threaded and not designed for concurrent load. In production we would switch to a proper **WSGI server** such as Gunicorn (`gunicorn -w 4 app:app`) to handle multiple workers. The entire service would be **containerized** with Docker so it can be versioned, reproduced, and deployed consistently across environments. Secrets and configuration (model paths, ports) would move into **environment variables** or a secrets manager rather than being hardcoded. The API would be placed behind **HTTPS** (via a reverse proxy like Nginx or a cloud load balancer) to encrypt traffic. Finally, an orchestrator like Kubernetes would manage autoscaling and health restarts.

### How would you handle model versioning?

Each trained model artifact would be tagged with a version number and stored in an artifact store (e.g., MLflow, S3 with versioned keys). A **shadow deployment** or **canary release** pattern would route a small percentage of traffic to the new model while the old one handles the rest. Automated evaluation checks would compare the new model's metrics (accuracy, AUC, latency) against a baseline before promoting it. If the new model is worse, the registry would not promote it and the old model stays live. A **rollback** mechanism means any version can be reactivated within seconds by pointing the service at a different artifact path.

### What monitoring would you add?

Three layers of monitoring matter most:
- **Operational**: request latency (p50/p95/p99), error rate, and throughput — these catch infrastructure problems.
- **Data quality**: input feature distributions compared to the training distribution (using tools like Evidently or Whylogs) to detect **input drift** before it degrades predictions silently.
- **Model quality**: where labels are available (via delayed feedback), track **prediction drift** and accuracy over time. Alert when any metric crosses a threshold.

### How would the architecture change at 1,000 requests per second?

At that scale the single-process Flask model becomes a bottleneck immediately. The service would need to run behind a **load balancer** distributing traffic across many Gunicorn instances (horizontally scaled). If inference is expensive, a **request queue** (e.g., Kafka or SQS) decouples spikes from processing capacity. For latency-sensitive use cases, the model could be served via a dedicated inference server (TorchServe, Triton, or SageMaker endpoints) that is optimized for throughput and supports batching internally. Caching repeated identical inputs (via Redis) can cut redundant inference entirely. The API layer itself would be kept stateless so any replica can serve any request.